# California Housing — Exploratory Data Analysis

This notebook explores the California Housing dataset used to train the House Price Predictor model. We'll analyze distributions, correlations, and patterns that inform the modeling approach.

## 1. Load Data & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

df = pd.read_csv('../data/train.csv')
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.info()
print("\n" + "="*50)
df.describe()

## 2. Missing Values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Missing values:")
    print(missing)
    sns.barplot(x=missing.values, y=missing.index, hue=missing.index, legend=False, palette='viridis')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values in numeric columns.")

# total_bedrooms has 207 NaNs - the model uses imputation

In [ ]:
print("Missing per column:")
print(df.isnull().sum())

## 3. Target Variable — Median House Value

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['median_house_value'] / 1000, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Median House Value ($1000s)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Target Variable')

df['median_house_value'].plot(kind='box', ax=axes[1])
axes[1].set_ylabel('Median House Value ($)')
axes[1].set_title('Box Plot')

plt.tight_layout()
plt.show()

print(f"Mean: ${df['median_house_value'].mean():,.0f}")
print(f"Median: ${df['median_house_value'].median():,.0f}")
print(f"Std: ${df['median_house_value'].std():,.0f}")

## 4. Geographic Distribution

In [ ]:
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['median_house_value'], cmap='YlOrRd', alpha=0.5, s=5
)
plt.colorbar(scatter, label='Median House Value ($)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('House Prices Across California (Coastal & Bay Area = Higher)')
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
numeric_cols = ['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=40, edgecolor='black', alpha=0.7)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')

plt.tight_layout()
plt.suptitle('Numeric Feature Distributions', y=1.02, fontsize=14)
plt.show()

## 6. Correlation Analysis

In [ ]:
corr = df[numeric_cols + ['median_house_value']].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.title('Correlation Matrix (Target: median_house_value)')
plt.tight_layout()
plt.show()

print("\nCorrelation with target (median_house_value):")
corr['median_house_value'].drop('median_house_value').sort_values(ascending=False)

## 7. Key Relationships with Target

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0,0].scatter(df['median_income'], df['median_house_value'], alpha=0.3, s=5)
axes[0,0].set_xlabel('Median Income')
axes[0,0].set_ylabel('Median House Value ($)')
axes[0,0].set_title('Income vs Price (strongest correlate)')

axes[0,1].scatter(df['housing_median_age'], df['median_house_value'], alpha=0.3, s=5)
axes[0,1].set_xlabel('Housing Median Age')
axes[0,1].set_ylabel('Median House Value ($)')
axes[0,1].set_title('Age vs Price')

axes[1,0].scatter((df['total_rooms']/df['households'].replace(0, np.nan)), df['median_house_value'], alpha=0.3, s=5)
axes[1,0].set_xlabel('Rooms per Household')
axes[1,0].set_ylabel('Median House Value ($)')
axes[1,0].set_title('Rooms/Household vs Price')

axes[1,1].scatter((df['total_bedrooms']/df['total_rooms'].replace(0, np.nan)), df['median_house_value'], alpha=0.3, s=5)
axes[1,1].set_xlabel('Bedrooms / Rooms')
axes[1,1].set_ylabel('Median House Value ($)')
axes[1,1].set_title('Bedroom Ratio vs Price')

plt.tight_layout()
plt.show()

## 8. Ocean Proximity (Categorical)

In [ ]:
if 'ocean_proximity' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    df['ocean_proximity'].value_counts().plot(kind='bar', ax=axes[0])
    axes[0].set_title('Count by Ocean Proximity')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=45)
    
    sns.boxplot(data=df, x='ocean_proximity', y='median_house_value', ax=axes[1])
    axes[1].set_title('Price by Ocean Proximity')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("ocean_proximity not in dataset")

## 9. Summary & Model Implications

**Key Findings:**

- **median_income** is the strongest predictor (correlation ~0.69)
- Geographic clustering: coastal/Bay Area = higher prices
- **total_bedrooms** has 207 missing values → model uses median imputation
- **ocean_proximity** is categorical (not used in current model; could be encoded)
- Target has long right tail → log transform could help linear models

**Model choices:** Gradient Boosting handles non-linearity and mixed feature types well, which aligns with these patterns.